# ACME Inc. -- Data Engineering POC
## Technical Documentation and Presentation Guide

Reference date: April 1, 2021 (last sale: March 30, 2021)
Platform: Microsoft Fabric -- Lakehouse with Medallion architecture
Scope: Proof of Concept -- Sales and Margin Analysis for a US-based clothing distributor

---

## 1. Architecture Overview

The solution follows the three-layer Medallion pattern on Microsoft Fabric OneLake:

`
Source files (CSV, XML, Excel)
        |
        v
[BRONZE]  -- Raw, immutable, auditable
   Files land as Delta tables. No transformations.
   Data preserved exactly as received from source.
        |
        v
[SILVER]  -- Clean, typed, trusted
   Correct types. Explicit null handling.
   Known source errors documented and fixed.
   Geographic and business enrichment applied.
   Invalid records sent to quarantine (bronze_quarantine).
        |
        v
[GOLD]    -- Star Schema, BI-ready
   Fully denormalized dimensions. Facts with computed metrics.
   Power BI semantic model connected via Direct Lake.
`

The Gold model is a Constellation Schema -- two fact tables sharing conformed dimensions:
- fact_sales_items: order line grain (17,032 rows)
- fact_orders: order grain (6,571 rows)
- fact_budget: employee x year grain (40 rows after unpivot)

---

## 2. Data Sources

| File | Format | Bronze Table | Rows |
|---|---|---|---|
| Orders.csv | CSV | bronze_orders | 6,571 |
| Order_Details.csv | CSV | bronze_order_details | 17,032 |
| Shipments01/02/03.csv | CSV (3 files) | bronze_shipments | 17,226 |
| Customers.csv | CSV | bronze_customers | 100 |
| Products.csv | CSV | bronze_products | 77 |
| Categories.csv | CSV | bronze_categories | 10 |
| Divisions.csv | CSV | bronze_divisions | 6 |
| Shippers.csv | CSV | bronze_shippers | 3 |
| Suppliers.xml | XML | bronze_suppliers | 29 |
| Employees Offices.xlsx | Excel (2 sheets) | bronze_employees + bronze_offices | 35 + 5 |
| Budget.xlsx | Excel | bronze_budget | 8 |

Excluded: SummaryReport.xls -- validation artifact, not a data source.

---

## 3. Tables by Layer

### Bronze (12 tables)
Exact mirror of sources. Schema inferred. _ingested_at on every table.

### Silver (12 tables)
| Table | Key changes |
|---|---|
| silver_orders | Dates parsed (M/d/yyyy), NULL EmployeeID to -1, fiscal periods derived |
| silver_order_details | Discount rounded (IEEE 754 fix), LineTotal computed, DiscountBand added |
| silver_shipments | Dates parsed, _data_warning added (legacy date issue) |
| silver_customers | DivisionID derived from Country, geographic enrichment, encoding cleanup |
| silver_products | MarginAbs, MarginPct, MarginBand, PriceTier, InventoryStatus added |
| silver_employees | "confidential" salary handled, Hire_Date typed, unknown member injected |
| silver_offices | OfficeStateProvince N/A for non-US, geographic enrichment |
| silver_suppliers | Typo fixed, geographic enrichment |
| silver_categories | "Weaher" typo fixed, placeholder description replaced |
| silver_divisions | Duplicate PK quarantined, Central America injected with ID=6 |
| silver_shippers | Unknown Shippers 4 and 5 injected (referenced in orders, missing in source) |
| silver_budget | Office forward-filled, unpivoted to long format, BudgetMonthly computed |

### Gold (9 tables)
| Table | Type | Grain |
|---|---|---|
| gold_fact_sales_items | Fact | Order line |
| gold_fact_orders | Fact | Order |
| gold_fact_budget | Fact | Employee x Year |
| gold_dim_date | Dimension | Day (2007-2021) |
| gold_dim_customer | Dimension | Customer |
| gold_dim_product | Dimension | Product |
| gold_dim_employee | Dimension | Employee |
| gold_dim_shipper | Dimension | Shipper |
| gold_dq_issues | DQ | One row per issue |

---

## 4. Data Quality Issues Found

### Critical
| Issue | Impact | Action taken |
|---|---|---|
| TotalOrder = 0 for all 6,571 orders | Revenue cannot come from this column | Recomputed from Order_Details |
| ShipmentDate (2007-2012) predates OrderDate (2016-2021) in all rows | Delivery SLA cannot be calculated | Flagged in _data_warning column |
| ShipperIDs 4 and 5 missing from Shippers.csv | 20% of orders have no shipper | Unknown Members injected |
| DivisionID=2 duplicated (North America AND Central America) | Customers incorrectly classified | Duplicate quarantined, Central America gets ID=6 |

### High
| Issue | Action taken |
|---|---|
| UnitPrice in order_details differs from catalog in 99.4% of rows | Expected -- historical transaction price vs current catalog (SCD). Documented. |
| Discount floating-point artifacts (0.050000001) | Rounded to 2 decimal places |
| Year_Salary = "confidential" for president (EmpID=2) | NULL + Salary_Confidential flag |
| Budget Office column has merged cells -- forward-fill needed | Forward-fill applied in Silver |

### Medium
| Issue | Action taken |
|---|---|
| EmployeeID NULL in 5 orders | Unknown Employee (-1) injected |
| Typo "Weaher" in Categories 9 and 10 | Fixed to "Weather" in Silver |
| Typo "Dressed for Succes" in Suppliers | Fixed to "Dressed for Success" in Silver |
| Windows-1252 encoding in CSV files (garbled city names) | Files read with windows-1252 encoding (v2 fix) |

---

## 5. Metrics Available in Power BI

All computed in gold_fact_sales:

### gold_fact_sales_items (line-level)

| Metric | Formula | Note |
|---|---|---|
| SalesAmount | Qty x HistoricalUnitPrice x (1 - Discount) | Uses transaction price, not catalog price |
| GrossMargin | SalesAmount - (Qty x UnitCost) | Uses CURRENT UnitCost -- known limitation if costs changed historically |
| GrossMarginPct | GrossMargin / SalesAmount x 100 | ROW-LEVEL ONLY. For aggregate, use DAX: DIVIDE(SUM(GrossMargin), SUM(SalesAmount)) |

### gold_fact_orders (order-level)

| Metric | Formula | Note |
|---|---|---|
| TotalProductSales | SUM(LineTotal) per order | Aggregated from order_details |
| TotalOrderValue | TotalProductSales + Freight | Includes shipping cost |
| DeliveryDays | ShipmentDate - OrderDate | Negative -- legacy data issue, see section 4 |

---

## 6. Business Questions -- Answers

Average sales per transaction and per customer:
Available in fact_sales. AVG(SalesAmount) grouped by OrderID gives per-transaction.
SUM grouped by CustomerID divided by COUNT DISTINCT OrderID gives per-customer.

Drill down from Product Group to Product to Customer:
Hierarchy in gold_dim_product: CategoryName (group) -> ProductName.
Joined to fact_sales via ProductID and to gold_dim_customer via CustomerID.

Quantity sold vs sales by Product, Region, Sales Rep, Supplier:
fact_sales contains both Quantity and SalesAmount. All required dimensions are available.

Customer count YTD vs LY YTD:
gold_dim_date has Year, FiscalYear, Month. Period intelligence via DAX TIME INTELLIGENCE in Power BI.

Delivery performance:
DeliveryDays is available in fact_sales but is negative for all rows.
The shipment data originates from a legacy system (2007-2012). Current SLA cannot be measured.
This finding requires immediate attention from the logistics team.

Why does UnitPrice appear in two tables?
Products.UnitPrice = the CURRENT catalog price.
Order_Details.UnitPrice = the HISTORICAL price at the time of the sale.
This is the Slowly Changing Dimension (SCD) pattern. Products.UnitPrice changes over time,
but each transaction must preserve the price that was actually charged.
In a production environment, Products should be modeled as SCD Type 2 with
Valid_From, Valid_To, and Is_Current columns to track full catalog price history.

Other duplicate data?
Yes. Shipments contains CustomerID, ProductID, and EmployeeID which already exist
in Orders and Order_Details. This redundancy is dangerous -- if they ever disagree,
it means a shipment was processed with different data than the original order.
Cross-validation checks are implemented in NB_04 to detect these divergences.

---

## 7. Consultant Recommendations

Immediate (before go-live):
1. Fix DivisionID=2 in the source file (Divisions.csv) -- Central America needs its own ID.
2. Align with the logistics team on Shipments data -- the 2007-2012 dates are unusable for current SLA.
3. Register ShipperIDs 4 and 5 in the shippers master data.
4. Clarify with Finance whether Freight is part of Gross Revenue or a pass-through cost.

Short term (next sprint):
5. Implement SCD Type 2 on the Products table to track catalog price history.
6. Integrate a more recent shipments data source to enable real delivery SLA analysis.
7. Migrate pipeline to incremental loads using MERGE/UPSERT for full idempotency.
8. Add automated alerts on bronze_quarantine for the governance team.

Business opportunities identified:
9. Seasonality: order concentration in specific months suggests inventory planning opportunity.
10. Inactive customers: 2 registered customers have never placed an order -- activation campaign opportunity.
11. Stockout risk: products with zero stock and open orders need immediate attention.
12. Supply chain concentration: geographic clustering of suppliers creates single-point-of-failure risk.

---

## 8. System Walkthrough

Notebook execution order:
1. NB_01_Bronze_Ingestion -- raw file ingestion
2. NB_02_Silver_Transformation -- cleansing and enrichment
3. NB_03_Gold_StarSchema -- star schema build
4. NB_04_DataQuality -- data quality validation

Orchestration:
Data Pipeline in Fabric with activities in dependency chain.
Each notebook runs only if the previous one completed successfully.

Semantic model:
ACME_SemanticModel connected to the Lakehouse via Direct Lake mode.
Relationships defined in the data model view.
ACME_Sales_Report contains 5 analysis pages covering sales, margin, products, delivery, and data quality.
